In [12]:
import polars as pl
import duckdb
import os

# 1. Load data dan duplikasi skala besar (~6.8 juta baris)
DATA_PATH = "../data/raw/train.csv"
df_original = pl.read_csv(DATA_PATH)
df = pl.concat([df_original] * 150)

# Simpan ke CSV lokal agar bisa dibaca DuckDB SQL dengan aman
os.makedirs("../output", exist_ok=True)
temp_csv_path = "../output/temp_train_large.csv"
df.write_csv(temp_csv_path)

print(f"Total baris saat ini: {df.height:,} baris (Memenuhi syarat Big Data >1 juta baris!)")

# 2. Inisialisasi koneksi DuckDB
con = duckdb.connect()

print("\n" + "="*50)
print("=== 1. ANALISIS UTAMA: PERFORMA BERDASARKAN KOTA ===")
print("="*50)
query_city = f"""
    SELECT 
        TRIM(City) AS City, 
        COUNT(ID) AS total_orders,
        ROUND(AVG(CAST(NULLIF(TRIM(Delivery_person_Age), 'NaN') AS FLOAT)), 2) AS avg_age,
        ROUND(AVG(CAST(NULLIF(TRIM(Delivery_person_Ratings), 'NaN') AS FLOAT)), 2) AS avg_rating
    FROM read_csv_auto('{temp_csv_path}')
    WHERE City IS NOT NULL AND TRIM(City) != 'NaN'
    GROUP BY TRIM(City)
    ORDER BY total_orders DESC
"""
df_city = pl.DataFrame(con.execute(query_city).fetchall(), schema=[desc[0] for desc in con.execute(query_city).description], orient="row")
display(df_city)


print("\n" + "="*50)
print("=== 2. ANALISIS LANJUTAN: Kondisi Cuaca & Kepadatan Lalu Lintas ===")
print("="*50)
query_weather = f"""
    SELECT 
        TRIM(Weatherconditions) AS weather,
        TRIM(Road_traffic_density) AS traffic_density,
        COUNT(ID) AS total_orders,
        ROUND(AVG(CAST(NULLIF(TRIM(Delivery_person_Ratings), 'NaN') AS FLOAT)), 2) AS avg_rating
    FROM read_csv_auto('{temp_csv_path}')
    WHERE Weatherconditions IS NOT NULL AND Road_traffic_density IS NOT NULL
    GROUP BY TRIM(Weatherconditions), TRIM(Road_traffic_density)
    ORDER BY total_orders DESC
    LIMIT 5
"""
df_weather = pl.DataFrame(con.execute(query_weather).fetchall(), schema=[desc[0] for desc in con.execute(query_weather).description], orient="row")
display(df_weather)


print("\n" + "="*50)
print("=== 3. ANALISIS LANJUTAN: Statistik Berdasarkan Jenis Kendaraan ===")
print("="*50)
query_vehicle = f"""
    SELECT 
        TRIM(Type_of_vehicle) AS vehicle_type,
        COUNT(ID) AS total_orders,
        ROUND(AVG(CAST(NULLIF(TRIM(Delivery_person_Age), 'NaN') AS FLOAT)), 2) AS avg_age,
        ROUND(AVG(CAST(NULLIF(TRIM(Delivery_person_Ratings), 'NaN') AS FLOAT)), 2) AS avg_rating
    FROM read_csv_auto('{temp_csv_path}')
    WHERE Type_of_vehicle IS NOT NULL AND TRIM(Type_of_vehicle) != 'NaN'
    GROUP BY TRIM(Type_of_vehicle)
    ORDER BY total_orders DESC
"""
df_vehicle = pl.DataFrame(con.execute(query_vehicle).fetchall(), schema=[desc[0] for desc in con.execute(query_vehicle).description], orient="row")
display(df_vehicle)

print("\nAnalisis EDA Multi-Dimensi DuckDB Berhasil Dijalankan Sempurna!")

Total baris saat ini: 6,838,950 baris (Memenuhi syarat Big Data >1 juta baris!)

=== 1. ANALISIS UTAMA: PERFORMA BERDASARKAN KOTA ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

City,total_orders,avg_age,avg_rating
str,i64,f64,f64
"""Metropolitian""",5113950,29.79,4.62
"""Urban""",1520400,28.88,4.67
"""Semi-Urban""",24600,32.57,4.49



=== 2. ANALISIS LANJUTAN: Kondisi Cuaca & Kepadatan Lalu Lintas ===


...


=== 3. ANALISIS LANJUTAN: Statistik Berdasarkan Jenis Kendaraan ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

vehicle_type,total_orders,avg_age,avg_rating
str,i64,f64,f64
"""motorcycle""",3965250,29.56,4.63
"""scooter""",2291400,29.53,4.64
"""electric_scooter""",572100,29.73,4.64
"""bicycle""",10200,31.33,3.33



Analisis EDA Multi-Dimensi DuckDB Berhasil Dijalankan Sempurna!


In [2]:
!pip install --upgrade --force-reinstall numpy plotly

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 4.5 MB/s  0:00:03 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 5.5 MB/s  0:00:01 eta 0:00:01
  Attempting uninstall: packaging
    Found existing installation: packaging 26.3
    Uninstalling packaging-26.3:
      Successfully uninstalled packaging-26.3
  Attempting uninstall: numpy
    Found existing installation: numpy None
    Can't uninstall 'numpy'. No files were found to uninstall.
  Attempting uninstall: narwhals━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/4 [numpy]
    Found existing installation: narwhals 2.26.0━━━━━━━━━━━━━━ 1/4 [numpy]
    Uninstalling narwhals-2.26.0:━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/4 [numpy]
      Successfully uninstalled narwhals-2.26.0━━━━━━━━━━━━━━━━ 1/4 [numpy]
  Attempting uninstall: plotly0m╺━━━━━━━━━━━━━━━━━━━ 2/4 [narwhals]
    Found existing installation: plotly 7.1.0━━━━━━━━━━━━━━━━━ 2/4 [narwhals]
    Uninstalling plotly-7.1.0:━━━━━━━━╺━━━━━━━━━ 3/4 [plotly]
   ━━━━━━━━━━━━━━━━━━━━

In [3]:
!pip install --user --upgrade --force-reinstall numpy plotly

  Using cached numpy-2.5.3-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
  Using cached plotly-7.1.0-py3-none-any.whl.metadata (8.8 kB)
  Using cached narwhals-2.26.0-py3-none-any.whl.metadata (15 kB)
  Using cached packaging-26.3-py3-none-any.whl.metadata (3.5 kB)
Using cached numpy-2.5.3-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (16.7 MB)
Using cached plotly-7.1.0-py3-none-any.whl (9.7 MB)
Using cached narwhals-2.26.0-py3-none-any.whl (474 kB)
Using cached packaging-26.3-py3-none-any.whl (129 kB)
   ━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/4 [numpy]  WARNING: The scripts f2py and numpy-config are installed in '/home/jovyan/.local/bin' which is not on PATH.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━ 3/4 [plotly]  WARNING: The script plotly_get_chrome is installed in '/home/jovyan/.local/bin' which is not on PATH.
  Con